# Cognifyz Technologies Data Science & Analytics Internship
## Level 1 & Core Restaurant Data Analytics Tasks

**Author:** Pallapu Ankamma Rao  
**Project:** Cognifyz Restaurant Data Analytics Dashboard  
**Dataset:** Official Cognifyz `Dataset.csv` (9,551 records, 21 columns)  

---
### Objectives Covered in this Notebook:
1. **Level 1 - Task 1: Top Cuisines Analysis**
   - Determine the top three most common cuisines.
   - Calculate the percentage of restaurants that serve each of the top cuisines.
2. **Level 1 - Task 2: City Analysis**
   - Identify the city with the highest number of restaurants.
   - Calculate the average rating for restaurants in each city.
   - Determine the city with the highest average rating.
3. **Level 1 - Task 3: Price Range Distribution**
   - Visualize the distribution of price ranges.
   - Calculate the percentage of restaurants in each price range category.
4. **Level 1 - Task 4: Online Delivery & Table Booking Analysis**
   - Determine the percentage of restaurants that offer online delivery.
   - Compare the average ratings of restaurants with and without online delivery.
   - Analyze whether restaurants across price ranges are more likely to offer table booking.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries loaded successfully!')

## 1. Data Ingestion & Sanitization
Load the official Cognifyz dataset and inspect the dimensions, column types, and missing values.

In [2]:
df = pd.read_csv('../data/Dataset.csv')
print('Dataset Shape:', df.shape)
print('Columns:', df.columns.tolist())

# Handle missing values in Cuisines
df['Cuisines'] = df['Cuisines'].fillna('Unknown / Not Specified').str.strip()
print('Missing values handled successfully!')

---
## Task 1: Top Cuisines Analysis
- Identify the top three cuisines
- Calculate the percentage of restaurants serving each

In [3]:
all_cuisines = []
for item in df['Cuisines'].dropna():
    for c in item.split(','):
        c_clean = c.strip()
        if c_clean and c_clean != 'Unknown / Not Specified':
            all_cuisines.append(c_clean)

cuisine_counts = pd.Series(all_cuisines).value_counts()
total_restaurants = len(df)
top_3 = cuisine_counts.head(3)

task1_summary = pd.DataFrame({
    'Cuisine': top_3.index,
    'Total Outlets': top_3.values,
    'Percentage (%)': (top_3.values / total_restaurants * 100).round(2)
})
print('=== TOP 3 CUISINES ===')
print(task1_summary)

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=task1_summary, x='Cuisine', y='Percentage (%)', palette='Blues_r', ax=ax)
ax.set_title('Top 3 Most Common Cuisines (% of Total Restaurants)', fontsize=14, pad=12)
ax.set_ylabel('Restaurant Share (%)')
ax.set_xlabel('Cuisine')
for idx, row in task1_summary.iterrows():
    ax.text(idx, row['Percentage (%)'] + 0.8, f"{row['Percentage (%)']}% ({int(row['Total Outlets']):,})", ha='center', fontweight='bold')
ax.set_ylim(0, task1_summary['Percentage (%)'].max() + 8)
plt.tight_layout()
plt.show()

**Task 1 Findings:**
1. **North Indian** is the single most dominant cuisine, offered by **3,960 restaurants (41.46%)**.
2. **Chinese** ranks second, served in **2,735 restaurants (28.64%)**.
3. **Fast Food** ranks third, present in **1,986 restaurants (20.79%)**.

---
## Task 2: City Analysis
- City with the highest restaurant density
- Average rating per city
- City with the highest average rating

In [4]:
city_counts = df['City'].value_counts()
print(f'City with Highest Number of Restaurants: {city_counts.index[0]} ({city_counts.iloc[0]:,} outlets)')

# City-level average rating (Rated restaurants > 0)
rated_df = df[df['Aggregate rating'] > 0]
city_ratings = rated_df.groupby('City')['Aggregate rating'].agg(['count', 'mean']).reset_index()
city_ratings.columns = ['City', 'Rated_Count', 'Avg_Rating']
city_ratings['Avg_Rating'] = city_ratings['Avg_Rating'].round(2)

# Cities with at least 10 rated restaurants for statistical robustness
robust_cities = city_ratings[city_ratings['Rated_Count'] >= 10].sort_values(by='Avg_Rating', ascending=False)
print('\nTop 5 Cities by Highest Average Rating (Min 10 outlets):')
print(robust_cities.head(5))

# Plot top 10 cities by volume
plt.figure(figsize=(10, 5))
city_counts.head(10).plot(kind='bar', color='teal')
plt.title('Top 10 Cities by Restaurant Count', fontsize=14, pad=12)
plt.xlabel('City')
plt.ylabel('Number of Restaurants')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Task 2 Findings:**
- **New Delhi** has the highest volume of restaurants: **5,473 outlets (57.3% of the entire dataset)**.
- The National Capital Region (New Delhi, Gurgaon, Noida, Faridabad) accounts for over 83% of the total dataset footprint.
- Cities with top average ratings include Pasig City (4.40), Quezon City (4.20), and Makati City (4.65).

---
## Task 3: Price Range Distribution
- Distribution of price ranges
- Percentage of restaurants in each range

In [5]:
price_counts = df['Price range'].value_counts().sort_index()
price_labels = {1: '1 - Budget', 2: '2 - Mid-Range', 3: '3 - Premium', 4: '4 - Fine Dining'}

price_df = pd.DataFrame({
    'Price Range': [price_labels[i] for i in price_counts.index],
    'Count': price_counts.values,
    'Percentage (%)': (price_counts.values / total_restaurants * 100).round(2)
})
print('=== PRICE RANGE BREAKDOWN ===')
print(price_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=price_df, x='Price Range', y='Percentage (%)', palette='viridis')
plt.title('Distribution of Restaurants by Price Tier', fontsize=14, pad=12)
plt.ylabel('Percentage of Restaurants (%)')
for idx, row in price_df.iterrows():
    plt.text(idx, row['Percentage (%)'] + 0.8, f"{row['Percentage (%)']}%\n({int(row['Count']):,})", ha='center', fontweight='bold')
plt.ylim(0, 55)
plt.tight_layout()
plt.show()

**Task 3 Findings:**
- **Price Range 1 (Budget)** represents **4,444 restaurants (46.53%)**.
- **Price Range 2 (Mid-Range)** represents **3,113 restaurants (32.59%)**.
- **Price Range 3 (Premium)** represents **1,408 restaurants (14.74%)**.
- **Price Range 4 (Fine Dining)** represents **586 restaurants (6.14%)**.

---
## Task 4: Online Delivery & Table Booking Analysis
- Online delivery availability percentage
- Rating differential (with vs without delivery)
- Table booking availability across price tiers

In [6]:
deliv_pct = (df['Has Online delivery'] == 'Yes').mean() * 100
print(f'Percentage offering Online Delivery: {deliv_pct:.2f}%')

# Rating comparison
rating_with_deliv = rated_df[rated_df['Has Online delivery'] == 'Yes']['Aggregate rating'].mean()
rating_without_deliv = rated_df[rated_df['Has Online delivery'] == 'No']['Aggregate rating'].mean()
print(f'Average Rating with Online Delivery: {rating_with_deliv:.2f}')
print(f'Average Rating without Online Delivery: {rating_without_deliv:.2f}')

# Table Booking by Price Tier
booking_by_price = pd.crosstab(df['Price range'], df['Has Table booking'], normalize='index') * 100
print('\nTable Booking Availability by Price Tier (%):')
print(booking_by_price.round(2))

# Plot table booking vs price range
booking_by_price.plot(kind='bar', stacked=True, color=['#ef4444', '#3b82f6'], figsize=(8, 4))
plt.title('Table Booking Availability Across Price Ranges', fontsize=14, pad=12)
plt.xlabel('Price Range (1: Budget -> 4: Fine Dining)')
plt.ylabel('Proportion (%)')
plt.legend(['No Table Booking', 'Offers Table Booking'], loc='upper left')
plt.tight_layout()
plt.show()

**Task 4 Findings:**
1. **Online Delivery Adoption:** Exactly **25.66% (2,451)** of restaurants offer online delivery.
2. **Customer Satisfaction Premium:** Restaurants with online delivery command a higher average rating (**3.48 vs. 3.42** among rated establishments).
3. **Table Booking Concentration:** Table booking is overwhelmingly concentrated in higher price tiers (**47.6% of tier 4** and **45.7% of tier 3** offer table booking, vs. **only 0.02% of tier 1**).